# 07 — Ablasi Angka: Apakah Batasan (Constraints) Membantu Prediksi Difficulty?

**Pertanyaan:** pipeline `text_clean` membuang semua angka (termasuk batasan seperti `10^5` atau `10^9`). Dugaan di rancangan: tingkat kesulitan sebenarnya bergantung pada batasan itu, sehingga membuangnya merugikan prediksi difficulty, tetapi **tidak** merugikan prediksi topik. Notebook ini **menguji** dugaan itu, bukan mengasumsikannya.

**Desain:** satu model tetap (TF-IDF × Logistic Regression, kombinasi tercepat dan juga yang terbaik di topik) dengan tiga variasi teks:

| Variasi | Isi |
|---|---|
| `clean` | `text_clean` (tanpa angka), **sama dengan Notebook 05/06** |
| `digits` | `text_clean_digits` (kata + angka dipertahankan) |
| `digits_only` | hanya token angka dari `text_clean_digits` (kata dibuang) |

Dua target: **difficulty** dan **topik** (kontrol: jika angka hanya membantu difficulty, perbaikannya tidak muncul di topik).

**Input (Add Data):** `data_clean.csv`, `labels.csv`, `split.csv`

**Output:** `tables/ablation_*.csv`

## Protokol (sama dengan Notebook 05/06)
Hyperparameter (`C`) dan threshold (topik) dipilih dari prediksi out-of-fold di data latih dengan fold dari `split.csv`; refit pada seluruh data latih; test set dipakai sekali. TF-IDF di-fit hanya pada data latih tiap fold.

## Detail penting
Pengaturan bawaan `TfidfVectorizer` **membuang token satu karakter**, sehingga `5` dari `10^5` hilang dan ablasi menjadi sia-sia. Karena itu di sini dipakai `token_pattern=r'(?u)\b\w+\b'`. Untuk variasi `clean` ini tidak mengubah apa pun (tidak ada token satu karakter di `text_clean`), jadi hasilnya harus **sama** dengan TF-IDF × LogReg di Notebook 05/06 (dicek di sel 5).

## 1. Import & Konfigurasi

In [1]:
import glob
import json
import os
import warnings

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, hamming_loss
from sklearn.multiclass import OneVsRestClassifier

warnings.filterwarnings('ignore')

SEED = 42
THR_GRID = np.round(np.arange(0.05, 0.96, 0.05), 2)
TFIDF_MAX = 10000
C_GRID = [0.1, 1.0, 10.0]
N_BOOT = 1000
VARIANTS = ['clean', 'digits', 'digits_only']

ON_KAGGLE = os.path.exists('/kaggle/input')
OUT_DIR = '/kaggle/working' if ON_KAGGLE else '.'
TAB_DIR = os.path.join(OUT_DIR, 'tables')
os.makedirs(TAB_DIR, exist_ok=True)


def find_file(name):
    if os.path.exists(name):
        return name
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    assert hits, f'{name} tidak ditemukan (tambahkan Kaggle Dataset-nya)'
    return hits[0]

## 2. Load Data dan Tiga Variasi Teks

In [2]:
df = pd.read_csv(find_file('data_clean.csv'))
labels = pd.read_csv(find_file('labels.csv'))
split = pd.read_csv(find_file('split.csv'))
assert (df['id'].values == labels['id'].values).all() and (df['id'].values == split['id'].values).all()

topic_cols = [c for c in labels.columns if c.startswith('topic_')]
TARGETS = {
    'difficulty': labels['difficulty_id'].values.astype(int),
    'topic': labels[topic_cols].values.astype(int),
}
N = len(df)

is_train = (split['set'] == 'train').values
train_idx = np.where(is_train)[0]
test_idx = np.where(~is_train)[0]
fold = split['fold'].values
folds = [(train_idx[fold[train_idx] != k], train_idx[fold[train_idx] == k]) for k in range(5)]

digits_text = df['text_clean_digits'].fillna('')
TEXTS = {
    'clean': df['text_clean'].fillna('').values,
    'digits': digits_text.values,
    'digits_only': digits_text.apply(lambda s: ' '.join(t for t in s.split() if t.isdigit())).values,
}

# Seberapa banyak angka yang ada?
all_tok = [t for s in digits_text for t in s.split()]
n_digit = sum(t.isdigit() for t in all_tok)
print(f'Porsi token angka: {n_digit / len(all_tok):.1%}  |  soal tanpa angka sama sekali: '
      f'{int((TEXTS["digits_only"] == "").sum())}')
print('Contoh digits_only:', TEXTS['digits_only'][0][:120])

Porsi token angka: 37.1%  |  soal tanpa angka sama sekali: 0
Contoh digits_only: 1 2 7 11 15 9 0 1 0 1 9 0 1 2 3 2 4 6 1 2 3 3 3 6 0 1 2 10 4 10 9 10 9 10 9 10 9 2


## 3. Pipeline

Vectorizer di-fit hanya pada data latih tiap fold (di-cache agar tidak dihitung ulang untuk tiap `C`). Probabilitas dikembalikan; prediksi topik memakai threshold global, prediksi difficulty memakai argmax.

In [3]:
def make_vec():
    return TfidfVectorizer(max_features=TFIDF_MAX, ngram_range=(1, 2), min_df=2,
                           sublinear_tf=True, token_pattern=r'(?u)\b\w+\b')


_vcache = {}


def get_xy(variant, tr, ev, key):
    ck = (variant, key)
    if ck not in _vcache:
        vec = make_vec()
        Xtr = vec.fit_transform(TEXTS[variant][tr])
        Xev = vec.transform(TEXTS[variant][ev])
        _vcache[ck] = (Xtr, Xev, vec)
    return _vcache[ck]


def fit_proba(target, Xtr, ytr, Xev, C):
    if target == 'topic':
        clf = OneVsRestClassifier(LogisticRegression(C=C, class_weight='balanced', max_iter=2000), n_jobs=-1)
    else:
        clf = LogisticRegression(C=C, class_weight='balanced', max_iter=2000)
    clf.fit(Xtr, ytr)
    return clf.predict_proba(Xev), clf


def macro_f1(yt, yp):
    return f1_score(yt, yp, average='macro', zero_division=0)


def predict(target, P, thr):
    return (P >= thr).astype(int) if target == 'topic' else P.argmax(axis=1)


def best_threshold(Yt, P):
    scores = [macro_f1(Yt, (P >= t).astype(int)) for t in THR_GRID]
    i = int(np.argmax(scores))
    return float(THR_GRID[i]), float(scores[i])


def run_variant(variant, target):
    Y = TARGETS[target]
    L = Y.shape[1] if Y.ndim == 2 else 3
    best = None
    for C in C_GRID:
        oof = np.zeros((N, L))
        for k, (tr, ev) in enumerate(folds):
            Xtr, Xev, _ = get_xy(variant, tr, ev, k)
            oof[ev] = fit_proba(target, Xtr, Y[tr], Xev, C)[0]
        if target == 'topic':
            thr, score = best_threshold(Y[train_idx], oof[train_idx])
        else:
            thr, score = None, macro_f1(Y[train_idx], oof[train_idx].argmax(axis=1))
        if best is None or score > best['score']:
            best = {'C': C, 'thr': thr, 'score': score, 'oof': oof.copy()}

    fold_scores = np.array([macro_f1(Y[ev], predict(target, best['oof'][ev], best['thr'])) for _, ev in folds])

    Xtr, Xte, vec = get_xy(variant, train_idx, test_idx, 'full')
    P_test, clf = fit_proba(target, Xtr, Y[train_idx], Xte, best['C'])
    pred = predict(target, P_test, best['thr'])
    yt = Y[test_idx]
    row = {
        'variant': variant, 'target': target, 'C': best['C'], 'threshold': best['thr'],
        'cv_macro_f1_mean': round(float(fold_scores.mean()), 4),
        'cv_macro_f1_std': round(float(fold_scores.std()), 4),
        'test_macro_f1': round(macro_f1(yt, pred), 4),
        'n_features': Xtr.shape[1],
    }
    if target == 'difficulty':
        row['test_accuracy'] = round(accuracy_score(yt, pred), 4)
    else:
        row['test_micro_f1'] = round(f1_score(yt, pred, average='micro', zero_division=0), 4)
        row['test_hamming'] = round(hamming_loss(yt, pred), 4)
    return row, {'pred': pred, 'fold_scores': fold_scores, 'clf': clf, 'vec': vec}

## 4. Jalankan (3 variasi × 2 target)

In [4]:
rows, store = [], {}
for target in ['difficulty', 'topic']:
    for variant in VARIANTS:
        row, extra = run_variant(variant, target)
        rows.append(row)
        store[(variant, target)] = extra
        print(f"[{target:10s} | {variant:11s}] CV={row['cv_macro_f1_mean']:.4f}±{row['cv_macro_f1_std']:.4f} "
              f"| test macro-F1={row['test_macro_f1']:.4f} | C={row['C']} thr={row['threshold']} "
              f"| fitur={row['n_features']}")

res = pd.DataFrame(rows)
res.to_csv(os.path.join(TAB_DIR, 'ablation_digits_results.csv'), index=False)
res

[difficulty | clean      ] CV=0.5008±0.0164 | test macro-F1=0.4955 | C=1.0 thr=None | fitur=10000
[difficulty | digits     ] CV=0.5450±0.0116 | test macro-F1=0.5457 | C=1.0 thr=None | fitur=10000
[difficulty | digits_only] CV=0.4947±0.0238 | test macro-F1=0.4592 | C=1.0 thr=None | fitur=2271
[topic      | clean      ] CV=0.5969±0.0093 | test macro-F1=0.6261 | C=10.0 thr=0.4 | fitur=10000
[topic      | digits     ] CV=0.5969±0.0103 | test macro-F1=0.6251 | C=10.0 thr=0.45 | fitur=10000
[topic      | digits_only] CV=0.3416±0.0068 | test macro-F1=0.3586 | C=1.0 thr=0.5 | fitur=2271


,variant,target,C,threshold,cv_macro_f1_mean,cv_macro_f1_std,test_macro_f1,n_features,test_accuracy,test_micro_f1,test_hamming
0,clean,difficulty,1.0,NaN,0.5008,0.0164,0.4955,10000,0.5069,NaN,NaN
1,digits,difficulty,1.0,NaN,0.5450,0.0116,0.5457,10000,0.5530,NaN,NaN
2,digits_only,difficulty,1.0,NaN,0.4947,0.0238,0.4592,2271,0.4608,NaN,NaN
3,clean,topic,10.0,0.40,0.5969,0.0093,0.6261,10000,NaN,0.6818,0.1049
4,digits,topic,10.0,0.45,0.5969,0.0103,0.6251,10000,NaN,0.6884,0.0978
5,digits_only,topic,1.0,0.50,0.3416,0.0068,0.3586,2271,NaN,0.4205,0.2636


## 5. Cek Reproduksi

Variasi `clean` harus menghasilkan angka **yang sama** dengan TF-IDF × LogReg di Notebook 05/06. Kalau berbeda, ada perbedaan pipeline dan seluruh ablasi ini tidak bisa dipercaya. Isi dua nilai acuan di bawah dari hasilmu sendiri (di Notebook 05/06 yang sudah kamu jalankan).

In [5]:
# Nilai acuan dari output Notebook 05 / 06 (TF-IDF x LogReg). Ganti bila hasilmu berbeda.
EXPECTED = {'difficulty': 0.4980, 'topic': 0.6234}

for target, expected in EXPECTED.items():
    got = float(res[(res['variant'] == 'clean') & (res['target'] == target)]['test_macro_f1'].iloc[0])
    status = 'OK' if abs(got - expected) < 0.002 else 'TIDAK SAMA, periksa pipeline'
    print(f'{target:10s}: ablasi={got:.4f}  acuan={expected:.4f}  -> {status}')

difficulty: ablasi=0.4955  acuan=0.4980  -> TIDAK SAMA, periksa pipeline
topic     : ablasi=0.6261  acuan=0.6234  -> TIDAK SAMA, periksa pipeline


## 6. Perbandingan Berpasangan terhadap `clean`

Dua cara memeriksa apakah angka menambah informasi:
1. **Per fold CV:** selisih Macro-F1 (`variant` − `clean`) di tiap 5 fold. Konsisten positif di semua fold = sinyal nyata.
2. **Bootstrap berpasangan di test set:** resampel baris yang **sama** untuk kedua variasi, hitung selisih Macro-F1, ambil interval 95%. Interval yang memuat 0 berarti selisihnya tidak bisa dibedakan dari noise.

In [6]:
rng = np.random.default_rng(SEED)
boot_idx = [rng.integers(0, len(test_idx), len(test_idx)) for _ in range(N_BOOT)]

cmp_rows = []
for target in ['difficulty', 'topic']:
    yt = TARGETS[target][test_idx]
    base_pred = store[('clean', target)]['pred']
    base_boot = np.array([macro_f1(yt[b], base_pred[b]) for b in boot_idx])
    base_cv = store[('clean', target)]['fold_scores']
    for variant in ['digits', 'digits_only']:
        pred = store[(variant, target)]['pred']
        v_boot = np.array([macro_f1(yt[b], pred[b]) for b in boot_idx])
        d = v_boot - base_boot
        d_cv = store[(variant, target)]['fold_scores'] - base_cv
        cmp_rows.append({
            'target': target, 'variant_vs_clean': variant,
            'delta_cv_mean': round(float(d_cv.mean()), 4),
            'folds_better': f'{int((d_cv > 0).sum())}/5',
            'delta_test': round(float(macro_f1(yt, pred) - macro_f1(yt, base_pred)), 4),
            'delta_test_ci_low': round(float(np.percentile(d, 2.5)), 4),
            'delta_test_ci_high': round(float(np.percentile(d, 97.5)), 4),
            'ci_excludes_0': bool(np.percentile(d, 2.5) > 0 or np.percentile(d, 97.5) < 0),
        })

cmp_df = pd.DataFrame(cmp_rows)
cmp_df.to_csv(os.path.join(TAB_DIR, 'ablation_digits_paired.csv'), index=False)
cmp_df

,target,variant_vs_clean,delta_cv_mean,folds_better,delta_test,delta_test_ci_low,delta_test_ci_high,ci_excludes_0
0,difficulty,digits,0.0442,5/5,0.0501,0.0187,0.0821,True
1,difficulty,digits_only,-0.0061,1/5,-0.0363,-0.0943,0.0155,False
2,topic,digits,0.0000,2/5,-0.0010,-0.0154,0.0130,False
3,topic,digits_only,-0.2552,0/5,-0.2675,-0.2943,-0.2407,True


## 7. Angka Apa yang Memengaruhi Difficulty?

Koefisien terbesar per kelas dari model `digits` (difficulty), dibatasi pada fitur yang mengandung angka. Ini memberi contoh konkret untuk pembahasan: batasan seperti apa yang diasosiasikan model dengan Easy atau Hard. **Ini asosiasi, bukan sebab**: model hanya melihat bahwa pola angka tertentu lebih sering muncul pada kelas tertentu.

In [7]:
clf = store[('digits', 'difficulty')]['clf']
names = np.array(store[('digits', 'difficulty')]['vec'].get_feature_names_out())
has_digit = np.array([any(ch.isdigit() for ch in n) for n in names])

out = {}
for ci, cname in enumerate(['Easy', 'Medium', 'Hard']):
    coef = clf.coef_[ci].copy()
    coef[~has_digit] = -np.inf
    top = np.argsort(-coef)[:12]
    out[cname] = [f'{names[i]} ({clf.coef_[ci][i]:.2f})' for i in top]
pd.DataFrame(out)

,Easy,Medium,Hard
0,100 (3.84),10 5 (2.11),2 1 (1.12)
1,length 100 (2.87),length 10 (1.31),3 3 (1.10)
2,100 1 (1.58),10 (1.18),20 (1.04)
3,50 (1.33),5 1 (0.87),1 3 (1.01)
4,nums 100 (1.24),8 (0.72),modulo 10 (0.99)
5,1000 (1.14),10 9 (0.67),length 1 (0.98)
6,50 1 (1.08),nums 10 (0.65),10 9 (0.95)
7,100 consists (1.04),0 index (0.62),5 10 (0.85)
8,length 50 (1.04),1 10 (0.60),1 4 (0.78)
9,1 100 (1.02),5 (0.56),7 1 (0.73)


## Cara membaca hasil (untuk Bab IV-C)

- **Angka membantu difficulty, tidak membantu topik:** dugaan rancangan didukung bila `delta` difficulty positif dengan interval bootstrap yang tidak memuat 0 dan konsisten di sebagian besar fold, sedangkan `delta` topik sekitar 0. Tulis sebagai temuan, lengkap dengan besar selisihnya.
- **Selisih kecil atau interval memuat 0:** dugaan tidak didukung data ini. Tulis apa adanya. "Batasan hilang saat preprocessing" lalu tidak boleh dijadikan penjelasan utama di paper, hanya sebagai kemungkinan yang tidak terbukti.
- **`digits_only` jauh di atas baseline acak:** berarti angka saja sudah membawa sinyal difficulty. Bandingkan dengan baseline Notebook 05 (Macro-F1 acak ≈ 0,336).
- **Keterbatasan:** token angka mencakup angka di contoh input/output, bukan hanya batasan, jadi sebagian "sinyal angka" bisa berasal dari ukuran contoh. Sebutkan ini.